# Supplementary Table 24 — TriSCOPE cross-cohort validation workbook

**Primary: SEA-AD validation** of the ROSMAP (Mathys 2019) discovery cohort across the three TriSCOPE modalities (predictive modeling, differential expression, pseudotime) — cross-cohort Fisher's exact overlap tests (background = 19,011 genes, BH-FDR per modality), genetically-anchored (coloc PP.H4>0.8) predictive-not-DE genes & their SEA-AD replication, SEA-AD pseudotime subcluster AUCs (CERAD & clinical AD), and SEA-AD tri-supported genes.

**Secondary: ROSMAP-427 validation** — original-vs-427 gene-predictor overlap and the cell-type × model test ROC-AUCs (genes / genes+APOE / genes+all-demographics / demographics-only).

Run top-to-bottom (uses the scenv kernel). Writes the .xlsx to `/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/`.

In [1]:
#!/usr/bin/env python3
# Supplementary Table 24 builder — TriSCOPE SEA-AD validation (primary) + ROSMAP-427 validation
import os, glob, numpy as np, pandas as pd, joblib
from collections import defaultdict
from scipy.stats import fisher_exact

# ============================== PATHS ==============================
SEAAD_ML   = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_seaad"
ROSMAP_ML  = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
R427_ML    = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_rosmap427"
SEAAD_DEG  = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results/Processed"
ROSMAP_DEG = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Clincal_batch_DE_Outputs_revision"
SEAAD_PT   = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_pseudotime/pseudotime_centroid/publication_summary_statistics.csv"
ROSMAP_PT  = "/n/groups/patel/adithya/Pseudotime_Outputs_HVG_final/publication_summary_statistics.csv"
COLOC_XLSX = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Coloc_Results.xlsx"
BG_PARQUET = "/home/adm808/NormalizedCellMatrixSyn18485175.parquet"

AUC_DIRS = {
 "genes":      "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_rosmap427",
 "genes_apoe": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_apoe_rosmap427",
 "genes_all":  "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_demographics_rosmap427",
 "demo":       "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_demographics_rosmap427",
}
OUT_DIR = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables"
OUT_XLSX = os.path.join(OUT_DIR, "Supplementary_Table_24_TriSCOPE_Validation.xlsx")
os.makedirs(OUT_DIR, exist_ok=True)

CANON = ["Ast", "Mic", "Inh", "Oli", "Opc", "Ex"]
PRETTY = {"Ast":"Astrocytes","Mic":"Microglia","Inh":"Inhibitory Neurons",
          "Oli":"Oligodendrocytes","Opc":"OPCs","Ex":"Excitatory Neurons"}
ROSMAP_CT = {"Ast":"Ast","Mic":"Mic","Inh":"In","Oli":"Oli","Opc":"Opc","Ex":"Ex"}   # ML/DEG/coloc
SEAAD_CT  = {"Ast":"Ast","Mic":"Mic","Inh":"Inh","Oli":"Oli","Opc":"Opc","Ex":"Ex"}
PT_PREFIX = {"Ast":"Ast","Mic":"Mic","Inh":"In","Oli":"Oli","Opc":"Opc","Ex":"Ex"}     # both datasets
SEAAD_DEG_FILES  = {ct:f"poisson_DE_results_SEAAD_{SEAAD_CT[ct]}_COMBINED.csv" for ct in CANON}
ROSMAP_DEG_FILES = {ct:f"poisson_DE_results_{ROSMAP_CT[ct]}.csv" for ct in CANON}

# ============================== HELPERS ==============================
def bh_fdr(p):
    p = np.asarray(p, float); n = p.size
    if n == 0: return p
    order = np.argsort(p)
    q = p[order] * n / np.arange(1, n+1)
    q = np.minimum.accumulate(q[::-1])[::-1]   # enforce monotonicity
    out = np.empty(n); out[order] = np.minimum(q, 1.0)   # place back to original positions
    return out

def ml_predictors(base, folder):
    counts = defaultdict(int)
    for s in range(1,6):
        p = os.path.join(base, folder, f"split_{s}", "maximal_classifier.joblib")
        if not os.path.exists(p): continue
        m = joblib.load(p)
        for g, imp in zip(m.feature_names_in_, m.feature_importances_):
            if imp > 0: counts[str(g)] += 1
    return {g for g,c in counts.items() if c >= 2}

def degs(base, fname):
    df = pd.read_csv(os.path.join(base, fname))
    df = df[(df["p_adj"]<0.05) & (df["log2FC"].abs()>0.25)]
    return set(df["gene"].astype(str))

def pt_top20(path, prefix):
    df = pd.read_csv(path)
    df["top_trajectory_genes"] = df["top_trajectory_genes"].fillna("").apply(lambda x:[g.strip() for g in x.split(",")])
    out = defaultdict(set)
    for _, row in df.iterrows():
        sub = row["Subcluster"]
        for ct, pref in prefix.items():
            if sub.startswith(pref):
                out[ct].update(row["top_trajectory_genes"][:20]); break
    return out


In [2]:
# ============================== LOAD ALL SETS ==============================
print("Loading ML predictors (SEA-AD, ROSMAP-old, ROSMAP-427)...")
seaad_ml  = {ct: ml_predictors(SEAAD_ML,  SEAAD_CT[ct])  for ct in CANON}
rosmap_ml = {ct: ml_predictors(ROSMAP_ML, ROSMAP_CT[ct]) for ct in CANON}
r427_ml   = {ct: ml_predictors(R427_ML,   SEAAD_CT[ct])  for ct in CANON}  # 427 folder uses Inh

print("Loading DEGs...")
seaad_deg  = {ct: degs(SEAAD_DEG,  SEAAD_DEG_FILES[ct])  for ct in CANON}
rosmap_deg = {ct: degs(ROSMAP_DEG, ROSMAP_DEG_FILES[ct]) for ct in CANON}

print("Loading pseudotime top-20...")
seaad_pt  = pt_top20(SEAAD_PT,  PT_PREFIX)
rosmap_pt = pt_top20(ROSMAP_PT, PT_PREFIX)

# ============================== BACKGROUND (== ml_triscope: ~19,011) ==============================
bg = set(pd.read_parquet(BG_PARQUET).index.astype(str))
for d in (rosmap_ml, seaad_ml, rosmap_deg, seaad_deg, rosmap_pt, seaad_pt):
    for s in d.values(): bg |= s
N_BG = len(bg)
print(f"Background universe N = {N_BG:,}")

def fisher_rows(rosmap_sets, seaad_sets, set1name="ROSMAP", set2name="SEA-AD"):
    rows = []
    for ct in CANON:
        r = rosmap_sets.get(ct, set()); s = seaad_sets.get(ct, set()); ov = r & s
        a,b,c = len(ov), len(r-ov), len(s-ov)
        N = len(bg | r | s); d = N-a-b-c
        OR,p = fisher_exact([[a,b],[c,d]], alternative="greater")
        rows.append({"CellType":PRETTY[ct], f"{set1name}_n":len(r), f"{set2name}_n":len(s),
                     "Overlap":a, "OddsRatio":OR, "PValue":p, "Overlap_Genes":", ".join(sorted(ov))})
    df = pd.DataFrame(rows); df["QValue_BH"] = bh_fdr(df["PValue"].values)
    cols = list(df.columns); cols.insert(cols.index("PValue")+1, cols.pop(cols.index("QValue_BH")))
    return df[cols]

de_overlap = fisher_rows(rosmap_deg, seaad_deg)
ml_overlap = fisher_rows(rosmap_ml,  seaad_ml)
pt_overlap = fisher_rows(rosmap_pt,  seaad_pt)


Loading ML predictors (SEA-AD, ROSMAP-old, ROSMAP-427)...
Loading DEGs...
Loading pseudotime top-20...
Background universe N = 19,011


In [3]:
# ============================== COLOC-ANCHORED PREDICTORS ==============================
# reconstruct coloc table from the per-cell-type sheets of Coloc_Results.xlsx
coloc_df = pd.concat(pd.read_excel(COLOC_XLSX, sheet_name=None).values(), ignore_index=True)
agg = coloc_df.groupby(["gene_symbol","cell_type"])["PP.H4.abf"].max().reset_index()
coloc_genes = {ct: set(agg[(agg["cell_type"]==ROSMAP_CT[ct]) & (agg["PP.H4.abf"]>0.8)]["gene_symbol"].astype(str)) for ct in CANON}

coloc_rows = []
for ct in CANON:
    rosmap_anchor = (coloc_genes[ct] & rosmap_ml[ct]) - rosmap_deg[ct]   # coloc + predictive, NOT DE (ROSMAP)
    replicated    = rosmap_anchor & seaad_ml[ct]                          # also predictive in SEA-AD
    coloc_rows.append({"CellType":PRETTY[ct],
                       "Coloc_PPH4>0.8_n":len(coloc_genes[ct]), "ROSMAP_ML_n":len(rosmap_ml[ct]),
                       "Coloc_and_Predictive_notDE":", ".join(sorted(rosmap_anchor)),
                       "Replicated_as_SEAAD_predictor":", ".join(sorted(replicated))})
coloc_tab = pd.DataFrame(coloc_rows)

# ============================== SEA-AD TRI-SUPPORTED ==============================
tri_rows = []
for ct in CANON:
    tri = seaad_ml[ct] & seaad_deg[ct] & seaad_pt.get(ct,set())
    tri_rows.append({"CellType":PRETTY[ct], "N_TriSupported":len(tri), "Genes":", ".join(sorted(map(str,tri)))})
tri_tab = pd.DataFrame(tri_rows)

# ============================== SEA-AD PSEUDOTIME AUCs ==============================
pt_auc = pd.read_csv(SEAAD_PT)
pt_auc = pt_auc.rename(columns={"mean_auc":"AD_AUC","mean_auc_cerad":"CERAD_AUC",
                                "combined_p_value":"AD_p","p_value_cerad":"CERAD_p",
                                "fdr_q_value":"AD_q","fdr_q_value_cerad":"CERAD_q"})
pt_auc_cols = ["Subcluster","n_cells","CERAD_AUC","auc_cerad_lower","auc_cerad_upper","CERAD_p","CERAD_q",
               "AD_AUC","auc_ci_lower","auc_ci_upper","AD_p","AD_q","top_trajectory_genes"]
pt_auc = pt_auc[[c for c in pt_auc_cols if c in pt_auc.columns]].sort_values("CERAD_AUC", ascending=False)

# ============================== ROSMAP-427 GENE VALIDATION (old vs 427 ML) ==============================
r427_gene = fisher_rows(rosmap_ml, r427_ml, set1name="ROSMAP_old", set2name="ROSMAP427")

# ============================== ROSMAP-427 AUC ==============================
def auc_of(p):
    df = pd.read_csv(p)
    for c in ["test_roc_auc","test_auc","roc_auc"]:
        if c in df.columns: return float(df[c].iloc[0])
    return np.nan
per_split = []
for model, base in AUC_DIRS.items():
    for ct in CANON:
        for s in range(1,6):
            p = os.path.join(base, ct, f"split_{s}", "output_csv.csv")
            if os.path.exists(p):
                per_split.append({"cell_type":PRETTY[ct],"model":model,"split":s,"test_roc_auc":auc_of(p)})
per_split_df = pd.DataFrame(per_split)
auc_summary = (per_split_df.groupby(["cell_type","model"])["test_roc_auc"]
               .agg(["mean","std","count"]).reset_index())
auc_summary["mean_pm_sd"] = auc_summary.apply(lambda r:f"{r['mean']:.3f} ± {r['std']:.3f} (n={int(r['count'])})",axis=1)
auc_pivot = auc_summary.pivot(index="cell_type", columns="model", values="mean_pm_sd")
auc_pivot = auc_pivot.reindex([PRETTY[c] for c in ["Ex","Inh","Ast","Oli","Mic","Opc"]])[["genes","genes_apoe","genes_all","demo"]].reset_index()


In [4]:
# ============================== VERIFY vs paragraph ==============================
print("\n--- VERIFY (should match the paragraph) ---")
chk = de_overlap.set_index("CellType")
for nm,row in [("Oligodendrocytes",None),("Excitatory Neurons",None),("Microglia",None)]:
    r = chk.loc[nm]; print(f"DE {nm}: overlap={r['Overlap']} OR={r['OddsRatio']:.2f} p={r['PValue']:.2e}")
ptc = pt_overlap.set_index("CellType")
print(f"PT Inh: overlap={ptc.loc['Inhibitory Neurons','Overlap']} OR={ptc.loc['Inhibitory Neurons','OddsRatio']:.2f} p={ptc.loc['Inhibitory Neurons','PValue']:.2e}")
print(f"Coloc ARL17B replicated cell types:", [r['CellType'] for _,r in coloc_tab.iterrows() if 'ARL17B' in r['Replicated_as_SEAAD_predictor']])
print(f"Coloc JAZF1 (Mic) anchor:", coloc_tab.set_index('CellType').loc['Microglia','Coloc_and_Predictive_notDE'])
print(f"PT AUC Ast1 CERAD:", pt_auc.set_index('Subcluster').loc['Ast1','CERAD_AUC'])

# ============================== WRITE WORKBOOK ==============================
title = (
 "SUPPLEMENTARY TABLE 24 — TriSCOPE cross-cohort validation.\n\n"
 "PRIMARY: SEA-AD validation of the ROSMAP (Mathys 2019) discovery cohort across the three TriSCOPE modalities "
 "(predictive modeling, differential expression, pseudotime), with cross-cohort Fisher's exact overlap tests "
 "(one-sided, background = "+f"{N_BG:,}"+" genes; BH-FDR per modality). Also includes genetically anchored "
 "(GWAS-eQTL colocalized, PP.H4>0.8) predictive-but-not-DE genes and their SEA-AD replication, the SEA-AD "
 "pseudotime subcluster AUCs (CERAD & clinical AD), and the SEA-AD tri-supported genes.\n\n"
 "SECONDARY: ROSMAP-427 (Mathys 2023 reprocessing) validation — cross-cohort gene-predictor overlap vs the "
 "original ROSMAP run, and the cell-type x model test ROC-AUCs (genes / genes+APOE / genes+all-demographics / "
 "demographics-only). ROSMAP-427 Ex/Inh/Oli used a 400 cells/donor cap (memory); Ast/Mic/Opc used all cells; "
 "demo used all cells.\n\n"
 "Predictor = LightGBM feature importance>0 in >=2/5 splits. DEG = p_adj<0.05 & |log2FC|>0.25. "
 "Pseudotime = top-20 trajectory genes aggregated per cell type."
)
with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as w:
    pd.DataFrame({"Description":[title]}).to_excel(w, sheet_name="Title_Page", index=False)
    de_overlap.to_excel(w, sheet_name="SEAAD_DE_Overlap", index=False)
    ml_overlap.to_excel(w, sheet_name="SEAAD_ML_Overlap", index=False)
    pt_overlap.to_excel(w, sheet_name="SEAAD_Pseudotime_Overlap", index=False)
    coloc_tab.to_excel(w, sheet_name="SEAAD_Coloc_Predictors", index=False)
    pt_auc.to_excel(w, sheet_name="SEAAD_Pseudotime_AUC", index=False)
    tri_tab.to_excel(w, sheet_name="SEAAD_TriSupported", index=False)
    r427_gene.to_excel(w, sheet_name="ROSMAP427_Gene_Validation", index=False)
    auc_pivot.to_excel(w, sheet_name="ROSMAP427_AUC", index=False)
    per_split_df.to_excel(w, sheet_name="ROSMAP427_AUC_per_split", index=False)
print(f"\nWROTE: {OUT_XLSX}")



--- VERIFY (should match the paragraph) ---
DE Oligodendrocytes: overlap=72 OR=6.33 p=7.22e-30
DE Excitatory Neurons: overlap=155 OR=2.36 p=5.02e-18
DE Microglia: overlap=19 OR=13.47 p=1.94e-14
PT Inh: overlap=20 OR=64.60 p=2.69e-27
Coloc ARL17B replicated cell types: ['Astrocytes', 'Microglia', 'Inhibitory Neurons', 'Oligodendrocytes', 'Excitatory Neurons']
Coloc JAZF1 (Mic) anchor: ARL17B, ERC2, JAZF1, RIN3, SYK, TMEM163, UBASH3B, USP6NL
PT AUC Ast1 CERAD: 0.9661700800306924

WROTE: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Revision/Tables/Supplementary_Table_24_TriSCOPE_Validation.xlsx
